In [ ]:
# import pandas as pd

# train_df = pd.read_json("data/processed/sft_dataset/train.jsonl", lines=True)
# val_df = pd.read_json("data/processed/sft_dataset/validation.jsonl", lines=True)
# # test_df = pd.read_json("data/processed/sft_dataset/test.jsonl", lines=True)

# print("Train:", train_df.shape)
# print("Validation:", val_df.shape)
# print("Test:", test_df.shape)

# display(train_df.head())

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

train_df = pd.read_json(
    "data/processed/sft_dataset/train.jsonl",
    lines=True
)

val_df = pd.read_json(
    "data/processed/sft_dataset/validation.jsonl",
    lines=True
)

test_df = pd.read_json(
    "data/processed/sft_dataset/test.jsonl",
    lines=True
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Tokenizer loaded successfully!")
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)

In [ ]:
print("Prompt:")
print(train_df.iloc[0]["prompt"][:500])

print("\nCompletion:")
print(train_df.iloc[0]["completion"][:500])

In [ ]:
def count_tokens(row):
    prompt_tokens = tokenizer(
        row["prompt"],
        add_special_tokens=False
    )["input_ids"]

    completion_tokens = tokenizer(
        row["completion"],
        add_special_tokens=False
    )["input_ids"]

    return len(prompt_tokens) + len(completion_tokens)


train_df["token_length"] = train_df.apply(count_tokens, axis=1)

print(train_df["token_length"].describe())
print("Maximum tokens:", train_df["token_length"].max())

In [ ]:
limits = [2048, 3072, 4096, 6144, 8192]

for limit in limits:
    count = (train_df["token_length"] > limit).sum()

    print(
        f"Above {limit}: {count} "
        f"({count / len(train_df) * 100:.2f}%)"
    )

In [ ]:
train_df["prompt_length"] = train_df["prompt"].apply(
    lambda x: len(tokenizer(x, add_special_tokens=False)["input_ids"])
)

train_df["completion_length"] = train_df["completion"].apply(
    lambda x: len(tokenizer(x, add_special_tokens=False)["input_ids"])
)

print("PROMPT LENGTH")
print(train_df["prompt_length"].describe())

print("\nCOMPLETION LENGTH")
print(train_df["completion_length"].describe())

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    gpu = torch.cuda.get_device_properties(0)

    print(
        "VRAM:",
        round(gpu.total_memory / (1024**3), 2),
        "GB"
    )
else:
    print("GPU not available. Enable GPU in Runtime settings.")

In [ ]:
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import prepare_model_for_kbit_training

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
import bitsandbytes as bnb
import torch

from transformers.utils import is_bitsandbytes_available

print("BitsAndBytes:", bnb.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# Clear cached availability check
is_bitsandbytes_available.cache_clear()

print(
    "BitsAndBytes detected:",
    is_bitsandbytes_available()
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Model loaded successfully!")

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

In [ ]:
model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

In [ ]:
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(name)
        break

In [ ]:
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["prompt", "completion"]].drop(columns=[], errors="ignore"),
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["prompt", "completion"]],
    preserve_index=False
)

print(train_dataset)
print(val_dataset)

In [ ]:
MAX_LENGTH = 3072

def fit_example(example):
    prompt = example["prompt"]
    completion = example["completion"]

    prompt_ids = tokenizer(
        prompt, add_special_tokens=False
    )["input_ids"]

    completion_ids = tokenizer(
        completion, add_special_tokens=False
    )["input_ids"]

    suffix = "<|im_end|>\n<|im_start|>assistant\n"
    suffix_ids = tokenizer(
        suffix, add_special_tokens=False
    )["input_ids"]

    if prompt_ids[-len(suffix_ids):] != suffix_ids:
        raise ValueError("Assistant generation marker is missing.")

    prompt_budget = MAX_LENGTH - len(completion_ids)

    if prompt_budget < len(suffix_ids) + 1:
        raise ValueError("Completion is too long.")

    if len(prompt_ids) > prompt_budget:
        body_ids = prompt_ids[:-len(suffix_ids)]
        prompt_ids = (
            body_ids[:prompt_budget - len(suffix_ids)]
            + suffix_ids
        )
        prompt = tokenizer.decode(
            prompt_ids, skip_special_tokens=False
        )

    return {"prompt": prompt, "completion": completion}


train_dataset = train_dataset.map(fit_example)
val_dataset = val_dataset.map(fit_example)

In [ ]:
def total_length(example):
    return (
        len(tokenizer(example["prompt"], add_special_tokens=False)["input_ids"])
        + len(tokenizer(example["completion"], add_special_tokens=False)["input_ids"])
    )

print("Maximum train length:", max(total_length(x) for x in train_dataset))
print("Maximum validation length:", max(total_length(x) for x in val_dataset))

In [ ]:
%pip install -q trl==1.17.1
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./resumeiq-qwen-lora",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    warmup_steps=15,
    lr_scheduler_type="cosine",

    fp16=False,
    bf16=False,

    optim="paged_adamw_8bit",

    max_length=3072,
    completion_only_loss=True,
    packing=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    eval_strategy="steps",
    eval_steps=200,

    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,

    logging_steps=10,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none"
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer
)

for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

print("Trainer ready!")

In [ ]:
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

print("Trainable parameters converted!")

In [ ]:
from collections import Counter

print(Counter(
    str(p.dtype)
    for p in trainer.model.parameters()
    if p.requires_grad
))
training_args.fp16 = False
training_args.bf16 = False

print("FP16:", training_args.fp16)
print("BF16:", training_args.bf16)

In [ ]:
import torch

torch.cuda.empty_cache()

training_result = trainer.train()

print("Training completed!")
print(training_result)

In [ ]:
%pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("GPU is not enabled!")

In [ ]:
SAVE_DIR = "/content/resumeiq-finetuned"

trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Model saved successfully!")

In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/content/resumeiq-finetuned",
    "zip",
    SAVE_DIR
)

print("ZIP created:", zip_path)

In [ ]:
from google.colab import files

files.download(zip_path)

In [ ]:
model = trainer.model
model.eval()

print("Fine-tuned model ready!")

In [ ]:
import os
import json
from pathlib import Path

# Project directory
PROJECT_DIR = Path.cwd()

print("Project directory:", PROJECT_DIR)

# Check adapter files
print("\nAdapter files:")

for filename in [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json"
]:
    path = PROJECT_DIR / filename
    print(filename, ":", path.exists())

# Check validation dataset
VALIDATION_PATH = PROJECT_DIR / "data" / "processed" / "validation.jsonl"

print("\nValidation file exists:", VALIDATION_PATH.exists())

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

print("Current directory:", PROJECT_DIR)

print("\nFiles and folders:")

for item in PROJECT_DIR.iterdir():
    print(item.name, "->", "Folder" if item.is_dir() else "File")

In [ ]:
from pathlib import Path

DATA_DIR = Path.cwd() / "data"

print("Data directory:", DATA_DIR)

for path in DATA_DIR.rglob("*"):
    if path.is_file():
        print("FILE:", path.relative_to(Path.cwd()))
    elif path.is_dir():
        print("FOLDER:", path.relative_to(Path.cwd()))

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

VALIDATION_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "sft_dataset"
    / "validation.jsonl"
)

print("Validation file exists:", VALIDATION_PATH.exists())
print("Validation path:", VALIDATION_PATH)

In [ ]:
import json

with open(VALIDATION_PATH, "r", encoding="utf-8") as f:
    validation_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Total validation records:", len(validation_records))

print("\nKeys:", validation_records[0].keys())

print("\nPrompt:")
print(validation_records[0]["prompt"][:1000])

print("\nExpected completion:")
print(validation_records[0]["completion"])

In [ ]:
for i, record in enumerate(validation_records[:5]):
    assert "prompt" in record, f"Missing prompt in record {i}"
    assert "completion" in record, f"Missing completion in record {i}"
    assert isinstance(record["prompt"], str)
    assert isinstance(record["completion"], str)

print("First 5 records passed structural validation.")

In [ ]:
import pandas as pd
import json

test_df = pd.read_json(
    "data/processed/sft_dataset/test.jsonl",
    lines=True
)

print("Dataset shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

print("\nFirst sample:")
print(json.dumps(test_df.iloc[0].to_dict(), indent=4))

print("\nMissing values:")
print(test_df.isnull().sum())

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH ="/content/resumeiq-finetuned"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()

print("Model loaded successfully!")
print("Device:", next(model.parameters()).device)

In [ ]:
import sys

!{sys.executable} -m pip install accelerate

In [ ]:
import sys
import accelerate

print("Python:", sys.executable)
print("Accelerate:", accelerate.__version__)
print("Location:", accelerate.__file__)

In [ ]:

import torch
import json

def predict_resume(resume_text):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert resume parser. "
                "Extract structured information from resumes "
                "and return ONLY valid JSON. "
                "Do not include explanations or extra text."
            )
        },
        {
            "role": "user",
            "content": f"Resume:\n{resume_text}"
        }
    ]

    # Apply the same Qwen chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract only newly generated tokens
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response


In [ ]:

resume_1 = """
John Smith
Software Engineer

Email: john@example.com

Experience:
Software Engineer at ABC Technologies
2022 - 2025

Developed REST APIs using Python and Flask.
Worked with MySQL and AWS.

Skills:
Python, Java, Flask, Django, MySQL, AWS

Education:
B.Tech Computer Science, XYZ University, 2022
"""


resume_2 = """
Priya Sharma
Computer Science Graduate

Education:
B.Tech Computer Science, ABC University, 2025

Skills:
Python, Machine Learning, SQL, Java, React

Projects:
Loan Approval Prediction:
Developed a classification model using Scikit-learn.

Student Management System:
Built using Flask and MySQL.

Certifications:
Python Programming
Machine Learning Fundamentals
"""


resume_3 = """
Alex Kumar

Professional Summary:
Motivated developer interested in backend development.

Technical Skills:
Python, FastAPI, PostgreSQL, Docker

Projects:
AI-powered Resume Screening System.
Developed a resume parser using NLP.

Education:
Bachelor of Technology

No contact information provided.
No employment history provided.
"""


test_resumes = {
    "Experienced Candidate": resume_1,
    "Fresher Candidate": resume_2,
    "Incomplete Resume": resume_3
}


In [ ]:

predictions = {}

for name, resume in test_resumes.items():

    print("\n", "=" * 60)
    print(name)
    print("=" * 60)

    response = predict_resume(resume)

    predictions[name] = response

    print(response)

    # Check JSON validity
    try:
        parsed = json.loads(response)
        print("\nJSON VALID: True")

    except json.JSONDecodeError as e:
        print("\nJSON VALID: False")
        print("Error:", e)
